<a href="https://colab.research.google.com/github/MEENAAI/ML-IDS-IPS-SMART-HOME/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [5]:
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42

# Load the anonymized dataset from this public repo.
DATA_URL = "https://raw.githubusercontent.com/MEENAAI/flyrank-ml-internship-starter/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(DATA_URL)

# Keep the same preparation logic used by the repo's reference pipeline.
df = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)].copy()
df = df.drop_duplicates(subset=["content_id"]).reset_index(drop=True)

# Target: declining trend.
df["is_declining_label"] = (
    df["trend_direction"].astype(str).str.lower().eq("down").astype(int)
)

# Numeric transformations used by the reference model.
df["log_impressions_90d"] = np.log1p(df["impressions_90d"])
df["log_clicks_90d"] = np.log1p(df["clicks_90d"])
df["log_sessions_90d"] = np.log1p(df["sessions_90d"])
df["log_ai_sessions_90d"] = np.log1p(df["ai_sessions_90d"])

numeric_features = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "log_impressions_90d",
    "log_clicks_90d",
    "log_sessions_90d",
    "log_ai_sessions_90d",
    "days_with_impressions",
    "days_with_sessions",
    "content_age_days",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct",
]

categorical_features = [
    "competition_level",
    "content_type",
    "main_intent",
    "age_tier",
    "freshness_tier",
    "word_count_tier",
    "impression_tier",
    "position_tier",
]

# Clean numeric features.
X_num = df[numeric_features].apply(pd.to_numeric, errors="coerce")
X_num = X_num.replace([np.inf, -np.inf], np.nan).fillna(0)

# One-hot encode categorical features.
X_cat = pd.get_dummies(
    df[categorical_features].fillna("unknown").astype(str),
    prefix=categorical_features,
    dtype=float,
)

X = pd.concat(
    [X_num.reset_index(drop=True), X_cat.reset_index(drop=True)],
    axis=1
)

y = df["is_declining_label"].astype(int)

print("Rows:", len(df))
print("Positive/declining rate:", round(y.mean(), 4))
print("Feature count:", X.shape[1])

Rows: 30000
Positive/declining rate: 0.5421
Feature count: 52


## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [2]:
# Client-aware 80/20 holdout, matching the reference pipeline.
all_indices = np.arange(len(df))
client_series = df["client_id"].fillna("unknown").astype(str)
unique_clients = client_series.drop_duplicates().to_numpy()

if len(unique_clients) >= 5:
    rng = np.random.default_rng(RANDOM_STATE)
    shuffled_clients = rng.permutation(unique_clients)

    test_client_count = max(1, int(round(len(shuffled_clients) * 0.20)))
    test_clients = set(shuffled_clients[:test_client_count])

    test_mask = client_series.isin(test_clients).to_numpy()

    train_idx = all_indices[~test_mask]
    test_idx = all_indices[test_mask]

    if (
        len(train_idx) > 0
        and len(test_idx) > 0
        and y.iloc[train_idx].nunique() == 2
        and y.iloc[test_idx].nunique() == 2
    ):
        split_strategy = "client_holdout"
    else:
        train_idx, test_idx = train_test_split(
            all_indices,
            test_size=0.20,
            random_state=RANDOM_STATE,
            stratify=y,
        )
        split_strategy = "stratified_row_holdout"
else:
    train_idx, test_idx = train_test_split(
        all_indices,
        test_size=0.20,
        random_state=RANDOM_STATE,
        stratify=y,
    )
    split_strategy = "stratified_row_holdout"

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]
y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

print("Split strategy:", split_strategy)
print("Train rows:", len(train_idx))
print("Test rows:", len(test_idx))
print("Train positive rate:", round(y_train.mean(), 4))
print("Test positive rate:", round(y_test.mean(), 4))

Split strategy: client_holdout
Train rows: 27675
Test rows: 2325
Train positive rate: 0.5548
Test positive rate: 0.391


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [3]:
def percentile_rank(series):
    return pd.to_numeric(series, errors="coerce").fillna(0).rank(
        method="average", pct=True
    )

def normalize(series):
    values = pd.to_numeric(series, errors="coerce").fillna(0)
    minimum = values.min()
    maximum = values.max()

    if maximum == minimum:
        return pd.Series(np.zeros(len(values)), index=values.index)

    return (values - minimum) / (maximum - minimum)


def precision_at_k(y_true, scores, k=50):
    temp = pd.DataFrame({
        "y": np.asarray(y_true),
        "score": np.asarray(scores)
    })

    top = temp.sort_values("score", ascending=False).head(k)

    return float(top["y"].mean())


# -----------------------------
# Week-4 baseline score
# -----------------------------
baseline = df.copy()

baseline["visibility_score"] = percentile_rank(
    np.log1p(baseline["impressions_90d"])
)

baseline["freshness_risk_score"] = percentile_rank(
    baseline["days_since_last_update"]
)

baseline["position_opportunity_score"] = (
    (1 - normalize(baseline["avg_position"].clip(lower=1, upper=50)))
    * baseline["visibility_score"]
    * (baseline["avg_position"] > 0).astype(int)
)

baseline["depth_gap_score"] = (
    (1 - percentile_rank(baseline["word_count"]))
    * baseline["visibility_score"]
)

baseline["baseline_refresh_score"] = (
    0.40 * baseline["visibility_score"]
    + 0.30 * baseline["freshness_risk_score"]
    + 0.25 * baseline["position_opportunity_score"]
    + 0.05 * baseline["depth_gap_score"]
).clip(0, 1)


baseline_test_scores = baseline.iloc[test_idx]["baseline_refresh_score"].to_numpy()

baseline_p50 = precision_at_k(
    y_test,
    baseline_test_scores,
    50
)


# -----------------------------
# Random Forest
# -----------------------------
model = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    min_samples_leaf=25,
    class_weight="balanced_subsample",
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

model.fit(X_train, y_train)

model_probabilities = model.predict_proba(X_test)[:, 1]

model_p50 = precision_at_k(
    y_test,
    model_probabilities,
    50
)


# Additional model metrics
model_predictions = (model_probabilities >= 0.5).astype(int)

accuracy = (model_predictions == y_test.to_numpy()).mean()
precision = precision_score(
    y_test,
    model_predictions,
    zero_division=0
)
recall = recall_score(
    y_test,
    model_predictions,
    zero_division=0
)
f1 = f1_score(
    y_test,
    model_predictions,
    zero_division=0
)
roc_auc = roc_auc_score(
    y_test,
    model_probabilities
)


comparison = pd.DataFrame({
    "Method": ["Week-4 baseline", "Random Forest"],
    "Precision@50": [baseline_p50, model_p50]
})

comparison

,Method,Precision@50
0,Week-4 baseline,0.24
1,Random Forest,0.74


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [4]:

# Build an error-analysis table for the test set.
error_frame = df.iloc[test_idx][
    [
        "content_id",
        "client_id",
        "trend_direction",
        "impressions_90d",
        "avg_position",
        "days_since_last_update",
        "word_count",
    ]
].copy()

error_frame["actual"] = y_test.to_numpy()
error_frame["model_probability"] = model_probabilities

error_frame["prediction"] = (
    error_frame["model_probability"] >= 0.5
).astype(int)

error_frame["error_type"] = "correct"

error_frame.loc[
    (error_frame["prediction"] == 1) &
    (error_frame["actual"] == 0),
    "error_type"
] = "false_positive"

error_frame.loc[
    (error_frame["prediction"] == 0) &
    (error_frame["actual"] == 1),
    "error_type"
] = "false_negative"

print("False positives:", (error_frame["error_type"] == "false_positive").sum())
print("False negatives:", (error_frame["error_type"] == "false_negative").sum())

print("\nHighest-confidence false positives:")
display(
    error_frame[
        error_frame["error_type"] == "false_positive"
    ].sort_values(
        "model_probability",
        ascending=False
    ).head(10)
)

print("\nHighest-confidence false negatives:")
display(
    error_frame[
        error_frame["error_type"] == "false_negative"
    ].sort_values(
        "model_probability",
        ascending=True
    ).head(10)
)


# Feature importance
importance = pd.DataFrame({
    "feature": X.columns,
    "importance": model.feature_importances_
}).sort_values(
    "importance",
    ascending=False
).head(15)

print("\nTop model features:")
display(importance)

False positives: 529
False negatives: 233

Highest-confidence false positives:


,content_id,client_id,trend_direction,impressions_90d,avg_position,days_since_last_update,word_count,actual,model_probability,prediction,error_type
23250,content_d2dffcc697a4,client_f74efabef1,stable,5091,14.1,20,4496.0,0,0.737130,1,false_positive
23559,content_00603b0349b4,client_f74efabef1,up,1076,25.6,20,2439.0,0,0.734944,1,false_positive
25913,content_331182ca4cae,client_f74efabef1,up,3026,35.9,20,3546.0,0,0.733631,1,false_positive
23750,content_e55b8ab078b0,client_f74efabef1,stable,369,21.8,20,2192.0,0,0.733059,1,false_positive
10155,content_643f585dc7f7,client_f74efabef1,up,761,25.1,20,1980.0,0,0.731120,1,false_positive
5966,content_f5013794ba57,client_f74efabef1,new,881,15.7,20,3622.0,0,0.730532,1,false_positive
28337,content_ea4417d89e2c,client_f74efabef1,stable,352,11.9,20,2556.0,0,0.729056,1,false_positive
4249,content_db1cd41b4b4f,client_f74efabef1,up,1482,12.9,105,2221.0,0,0.729052,1,false_positive
21530,content_b15a8dbdf66f,client_f74efabef1,up,1647,22.4,20,4095.0,0,0.727853,1,false_positive
2380,content_96da95476e63,client_f74efabef1,stable,784,7.4,20,2598.0,0,0.724807,1,false_positive



Highest-confidence false negatives:


,content_id,client_id,trend_direction,impressions_90d,avg_position,days_since_last_update,word_count,actual,model_probability,prediction,error_type
5770,content_28b4223f4e5f,client_98a3ab7c34,down,1,0.0,1,3109.0,1,0.079867,0,false_negative
3879,content_34b14c00f80c,client_d4735e3a26,down,3,0.0,20,659.0,1,0.082196,0,false_negative
27177,content_79ac977c6e0b,client_f74efabef1,down,3,0.7,8,2304.0,1,0.149546,0,false_negative
22991,content_472ce7ae14c0,client_d4735e3a26,down,3,0.3,20,684.0,1,0.152184,0,false_negative
5608,content_a55d958ec725,client_d4735e3a26,down,3,2.7,20,837.0,1,0.163987,0,false_negative
12864,content_f1ef151d5e36,client_d4735e3a26,down,3,2.0,20,978.0,1,0.165946,0,false_negative
12076,content_230de4c50860,client_d4735e3a26,down,3,2.0,20,824.0,1,0.169816,0,false_negative
25838,content_cbc3b52a2ac1,client_98a3ab7c34,down,2,3.0,1,2286.0,1,0.171345,0,false_negative
13659,content_4c437dd8c1ee,client_d4735e3a26,down,3,3.0,20,840.0,1,0.174066,0,false_negative
23810,content_37804210415c,client_d4735e3a26,down,4,2.0,20,813.0,1,0.174828,0,false_negative



Top model features:


,feature,importance
9,days_with_impressions,0.134951
5,log_impressions_90d,0.129377
14,avg_position,0.109203
11,content_age_days,0.092048
4,char_count,0.038676
32,age_tier_365+,0.036847
6,log_clicks_90d,0.036572
3,word_count,0.035406
13,ctr,0.035156
16,scroll_rate,0.033876


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.